<a href="https://colab.research.google.com/github/angelinephilo/nasaspaceapps26/blob/main/Future_Climate_Projection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install requests pandas xarray s3fs netCDF4 urllib3

In [2]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import pandas as pd
from multiprocessing.pool import ThreadPool

import xarray as xr
import s3fs
import netCDF4
from netCDF4 import Dataset

import os
import os.path

print("All libraries imported successfully!")
print("xarray version:", xr.__version__)
print("s3fs version:", s3fs.__version__)
print("netCDF4 version:", netCDF4.__version__)

All libraries imported successfully!
xarray version: 2025.12.0
s3fs version: 2026.9.0
netCDF4 version: 1.7.5.1.post1


In [3]:
##Connecting to the aws server type shit
fs = s3fs.S3FileSystem(
    anon=True,
    client_kwargs={'region_name': 'us-west-2'} ##us-west-2 is where the host server is
)

print("Connected to NASA S3!")
bucket = 'nex-gddp-cmip6'

##Checking available folders
contents = fs.ls(bucket, detail=False)

print("Available folders:")
for item in contents:
    print(item)

##Looking at the sub-files inside one folder
dataset_path = 'nex-gddp-cmip6/NEX-GDDP-CMIP6'
dataset_contents = fs.ls(dataset_path, detail=False)
print("Available folders and files:")

for item in dataset_contents:
    print(item)

model_path = dataset_path + '/ACCESS-CM2'
model_contents = fs.ls(model_path, detail=False)
print("ACCESS-CM2 contents:")

for item in model_contents:
    print(item)


Connected to NASA S3!
Available folders:
nex-gddp-cmip6/NEX-GDDP-CMIP6
nex-gddp-cmip6/NEX-GDDP-CMIP6-Tech_Note.pdf
nex-gddp-cmip6/NEX-GDDP-CMIP6-v2-Tech_Note.pdf
nex-gddp-cmip6/index.html
nex-gddp-cmip6/index_md5.txt
nex-gddp-cmip6/index_md5.txt.md5
nex-gddp-cmip6/index_v1.1_md5.txt
nex-gddp-cmip6/index_v1.1_md5.txt.md5
nex-gddp-cmip6/index_v1.2_md5.txt
nex-gddp-cmip6/index_v1.2_md5.txt.md5
nex-gddp-cmip6/index_v2.0_md5.txt
nex-gddp-cmip6/index_v2.0_md5.txt.md5
Available folders and files:
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-CM2
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-ESM1-5
nex-gddp-cmip6/NEX-GDDP-CMIP6/BCC-CSM2-MR
nex-gddp-cmip6/NEX-GDDP-CMIP6/CESM2
nex-gddp-cmip6/NEX-GDDP-CMIP6/CESM2-WACCM
nex-gddp-cmip6/NEX-GDDP-CMIP6/CMCC-CM2-SR5
nex-gddp-cmip6/NEX-GDDP-CMIP6/CMCC-ESM2
nex-gddp-cmip6/NEX-GDDP-CMIP6/CNRM-CM6-1
nex-gddp-cmip6/NEX-GDDP-CMIP6/CNRM-ESM2-1
nex-gddp-cmip6/NEX-GDDP-CMIP6/CanESM5
nex-gddp-cmip6/NEX-GDDP-CMIP6/EC-Earth3
nex-gddp-cmip6/NEX-GDDP-CMIP6/EC-Earth3-Veg-LR
nex-gddp-

In [4]:
##inspect file structure for each ssp model
scenario_path = dataset_path + '/ACCESS-CM2/ssp126'
scenario_contents = fs.ls(scenario_path, detail=False)

print("SSP126 contents:")

for item in scenario_contents:
    print(item)

ensemble_path = scenario_contents[0]
ensemble_contents = fs.ls(ensemble_path, detail=False)
print("Ensemble contents:")

for item in ensemble_contents:
    print(item)

##select one variable to inspect
variable_path = (
    dataset_path +
    '/ACCESS-CM2/ssp126/r1i1p1f1/tasmax'
)

##list available NetCDF files
files = fs.ls(variable_path, detail=False)
print("Number of files:", len(files))
print("\nFirst 10 files:")
for file in files[:10]:
    print(file.split('/')[-1])

##Extract years
import re
years = []

for file in files:
    filename = file.split('/')[-1]
    match = re.search(r'_(\d{4})\.nc$', filename)
    if match:
        years.append(int(match.group(1)))

years.sort()

print("Earliest year:", min(years))
print("Latest year:", max(years))
print("Total annual files:", len(years))

#separate v2.0 files from original files
v2_files = [
    file for file in files
    if file.endswith('_v2.0.nc')
]

original_files = [
    file for file in files
    if file.endswith('.nc')
    and not file.endswith('_v2.0.nc')
]

print("Original files:", len(original_files))
print("V2.0 files:", len(v2_files))

# Extract years from v2.0 files
v2_years = sorted([
    int(re.search(r'_(\d{4})_v2\.0\.nc$', file).group(1))
    for file in v2_files
])

print("V2.0 earliest year:", min(v2_years))
print("V2.0 latest year:", max(v2_years))
print("Total V2.0 years:", len(v2_years))





SSP126 contents:
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-CM2/ssp126/r1i1p1f1
Ensemble contents:
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-CM2/ssp126/r1i1p1f1/hurs
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-CM2/ssp126/r1i1p1f1/huss
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-CM2/ssp126/r1i1p1f1/pr
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-CM2/ssp126/r1i1p1f1/rlds
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-CM2/ssp126/r1i1p1f1/rsds
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-CM2/ssp126/r1i1p1f1/sfcWind
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-CM2/ssp126/r1i1p1f1/tas
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-CM2/ssp126/r1i1p1f1/tasmax
nex-gddp-cmip6/NEX-GDDP-CMIP6/ACCESS-CM2/ssp126/r1i1p1f1/tasmin
Number of files: 172

First 10 files:
tasmax_day_ACCESS-CM2_ssp126_r1i1p1f1_gn_2015.nc
tasmax_day_ACCESS-CM2_ssp126_r1i1p1f1_gn_2015_v2.0.nc
tasmax_day_ACCESS-CM2_ssp126_r1i1p1f1_gn_2016.nc
tasmax_day_ACCESS-CM2_ssp126_r1i1p1f1_gn_2016_v2.0.nc
tasmax_day_ACCESS-CM2_ssp126_r1i1p1f1_gn_2017.nc
tasmax_day_ACCESS-CM2_ssp126_r1i1p1f1_gn_2017_v2.0.n

In [5]:
# Get all climate model folders
model_paths = fs.ls(dataset_path, detail=False)
models = [
    path.split('/')[-1]
    for path in model_paths
]
print("Total climate models:", len(models))
for i, model in enumerate(models, 1):
    print(i, model)


Total climate models: 35
1 ACCESS-CM2
2 ACCESS-ESM1-5
3 BCC-CSM2-MR
4 CESM2
5 CESM2-WACCM
6 CMCC-CM2-SR5
7 CMCC-ESM2
8 CNRM-CM6-1
9 CNRM-ESM2-1
10 CanESM5
11 EC-Earth3
12 EC-Earth3-Veg-LR
13 FGOALS-g3
14 GFDL-CM4
15 GFDL-CM4_gr2
16 GFDL-ESM4
17 GISS-E2-1-G
18 HadGEM3-GC31-LL
19 HadGEM3-GC31-MM
20 IITM-ESM
21 INM-CM4-8
22 INM-CM5-0
23 IPSL-CM6A-LR
24 KACE-1-0-G
25 KIOST-ESM
26 MIROC-ES2L
27 MIROC6
28 MPI-ESM1-2-HR
29 MPI-ESM1-2-LR
30 MRI-ESM2-0
31 NESM3
32 NorESM2-LM
33 NorESM2-MM
34 TaiESM1
35 UKESM1-0-LL


In [6]:
##Check data availability
scenarios = ["historical", "ssp126", "ssp585"]
variables = ["tasmax", "tasmin", "pr", "hurs"]
inventory = []

for model in models:
    for scenario in scenarios:
        scenario_path = f"{dataset_path}/{model}/{scenario}"
        try:
            members = fs.ls(scenario_path, detail=True)
        except FileNotFoundError:
            continue
        # Identify ensemble member folders, e.g. r1i1p1f1
        member_paths = [
            item["name"]
            for item in members
            if item.get("type") == "directory"
        ]
        for member_path in member_paths:
            member = member_path.split("/")[-1]
            for variable in variables:
                variable_path = f"{member_path}/{variable}"
                try:
                    files = fs.ls(variable_path, detail=False)
                except FileNotFoundError:
                    files = []
                nc_files = [
                    f for f in files
                    if f.endswith(".nc")
                ]
                # Only use consistently versioned v2.0 files
                v2_files = [
                    f for f in nc_files
                    if f.endswith("_v2.0.nc")
                ]
                # Extract years from v2.0 filenames
                years = sorted([
                    int(match.group(1))
                    for f in v2_files
                    if (match := re.search(r"_(\d{4})_v2\.0\.nc$", f))
                ])
                inventory.append({
                    "model": model,
                    "scenario": scenario,
                    "member": member,
                    "variable": variable,
                    "v2_files": len(v2_files),
                    "first_year": min(years) if years else None,
                    "last_year": max(years) if years else None,
                    "number_of_years": len(years)
                })

inventory_df = pd.DataFrame(inventory)
inventory_df.head(10)

##summarise availability of data
summary = (
    inventory_df
    .groupby(["scenario", "variable"])
    .agg(
        models_available=("model", "nunique"),
        total_v2_files=("v2_files", "sum")
    )
    .reset_index()
)
print(summary)

##identify missing data
missing = inventory_df[inventory_df["v2_files"] == 0]
print("Missing model-variable combinations:", len(missing))
missing[["model", "scenario", "member", "variable"]]







      scenario variable  models_available  total_v2_files
0   historical     hurs                35            2145
1   historical       pr                35            2275
2   historical   tasmax                35            2015
3   historical   tasmin                35            2015
4       ssp126     hurs                32            2578
5       ssp126       pr                32            2751
6       ssp126   tasmax                32            2494
7       ssp126   tasmin                32            2494
8       ssp585     hurs                35            2837
9       ssp585       pr                35            3009
10      ssp585   tasmax                35            2666
11      ssp585   tasmin                35            2666
Missing model-variable combinations: 28


,model,scenario,member,variable
27,BCC-CSM2-MR,historical,r1i1p1f1,hurs
31,BCC-CSM2-MR,ssp126,r1i1p1f1,hurs
35,BCC-CSM2-MR,ssp585,r1i1p1f1,hurs
36,CESM2,historical,r4i1p1f1,tasmax
37,CESM2,historical,r4i1p1f1,tasmin
40,CESM2,ssp126,r4i1p1f1,tasmax
41,CESM2,ssp126,r4i1p1f1,tasmin
44,CESM2,ssp585,r4i1p1f1,tasmax
45,CESM2,ssp585,r4i1p1f1,tasmin
48,CESM2-WACCM,historical,r3i1p1f1,tasmax


In [7]:
##convert inventory into a table showing whether each variable exists
availability = (
    inventory_df
    .assign(available=inventory_df["v2_files"] > 0)
    .pivot_table(
        index=["model", "scenario", "member"],
        columns="variable",
        values="available",
        aggfunc="any",
        fill_value=False
    )
    .reset_index()
)

##check whether each model-member has all four variables
availability["complete"] = availability[variables].all(axis=1)

print("Complete model-member combinations:")
print(
    availability[availability["complete"]]
    [["model", "scenario", "member"]]
    .to_string(index=False)
)

##check which models have at least one complete member per scenario
model_scenario = (
    availability
    .groupby(["model", "scenario"])["complete"]
    .any()
    .unstack(fill_value=False)
)

print("Complete models per scenario:")
print(model_scenario.sum())

##models with complete data across all three scenarios
common_models = model_scenario[
    model_scenario[scenarios].all(axis=1)
].index.tolist()

print("\nModels complete across all three scenarios:", len(common_models))
print(common_models)

##find ensemble members that have all four variables and are available across all three scenarios
consistent_members = {}
for model in common_models:
    members_by_scenario = []
    for scenario in scenarios:
        members = set(
            availability.loc[
                (availability["model"] == model) &
                (availability["scenario"] == scenario) &
                (availability["complete"]),
                "member"
            ]
        )
        members_by_scenario.append(members)
    shared_members = set.intersection(*members_by_scenario)
    if shared_members:
        consistent_members[model] = sorted(shared_members)

print("Models with a shared ensemble member:", len(consistent_members))

for model, members in consistent_members.items():
    print(model, ":", members)

Complete model-member combinations:
           model   scenario   member
      ACCESS-CM2 historical r1i1p1f1
      ACCESS-CM2     ssp126 r1i1p1f1
      ACCESS-CM2     ssp585 r1i1p1f1
   ACCESS-ESM1-5 historical r1i1p1f1
   ACCESS-ESM1-5     ssp126 r1i1p1f1
   ACCESS-ESM1-5     ssp585 r1i1p1f1
       CMCC-ESM2 historical r1i1p1f1
       CMCC-ESM2     ssp126 r1i1p1f1
       CMCC-ESM2     ssp585 r1i1p1f1
      CNRM-CM6-1 historical r1i1p1f2
      CNRM-CM6-1     ssp126 r1i1p1f2
      CNRM-CM6-1     ssp585 r1i1p1f2
     CNRM-ESM2-1 historical r1i1p1f2
     CNRM-ESM2-1     ssp126 r1i1p1f2
     CNRM-ESM2-1     ssp585 r1i1p1f2
         CanESM5 historical r1i1p1f1
         CanESM5     ssp126 r1i1p1f1
         CanESM5     ssp585 r1i1p1f1
       EC-Earth3 historical r1i1p1f1
       EC-Earth3     ssp126 r1i1p1f1
       EC-Earth3     ssp585 r1i1p1f1
EC-Earth3-Veg-LR historical r1i1p1f1
EC-Earth3-Veg-LR     ssp126 r1i1p1f1
EC-Earth3-Veg-LR     ssp585 r1i1p1f1
       FGOALS-g3 historical r3i1p1f1
  

In [12]:
##inspecting all models, scenarios and variables (parallel processing)

from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
def inspect_file(model, scenario, variable):
    member = consistent_members[model][0]
    variable_path = (
        f"{dataset_path}/{model}/{scenario}/"
        f"{member}/{variable}"
    )
    files = fs.ls(variable_path, detail=False)
    v2_files = [
        f for f in files
        if f.endswith("_v2.0.nc")
    ]
    if not v2_files:
        return None
    sample_file = sorted(v2_files)[0]
    with fs.open(sample_file, "rb") as f:
        with xr.open_dataset(f, engine="h5netcdf") as ds:
            lat = ds["lat"].values.copy()
            lon = ds["lon"].values.copy()
            return {
                "model": model,
                "scenario": scenario,
                "variable": variable,
                "latitude_count": len(lat),
                "longitude_count": len(lon),
                "time_steps": ds.sizes["time"],
                "time_start": str(ds.time.values[0]),
                "time_end": str(ds.time.values[-1]),
                "calendar": (
                    ds.time.encoding.get("calendar")
                    or ds.time.attrs.get("calendar")
                ),
                "units": ds[variable].attrs.get("units"),
                "lat": lat,
                "lon": lon
            }
#generate all 324 inspection tasks
tasks = [
    (model, scenario, variable)
    for model in common_models
    for scenario in scenarios
    for variable in variables
]
inspection_results = []
errors = []
#inspect files in parallel
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = {
        executor.submit(inspect_file, *task): task
        for task in tasks
    }
    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            result = future.result()
            if result is not None:
                inspection_results.append(result)
        except Exception as e:
            errors.append({
                "model": futures[future][0],
                "scenario": futures[future][1],
                "variable": futures[future][2],
                "error": str(e)
            })
# Separate metadata and spatial coordinates
inspection_df = pd.DataFrame(inspection_results)
coordinate_data = {}
for _, row in inspection_df.iterrows():
    key = (row["model"], row["scenario"], row["variable"])
    coordinate_data[key] = (
        row["lat"],
        row["lon"]
    )
inspection_df = inspection_df.drop(columns=["lat", "lon"])

print("Total combinations expected:", len(tasks))
print("Total files inspected:", len(inspection_df))
print("Total errors:", len(errors))

if errors:
    print(pd.DataFrame(errors).to_string(index=False))

  0%|          | 0/324 [00:00<?, ?it/s]

Total combinations expected: 324
Total files inspected: 324
Total errors: 0


In [13]:
##compare spatial & temporal resolution

#compare metadata separately for each scenario and variable
metadata_summary = (
    inspection_df
    .groupby(["scenario", "variable"])
    .agg(
        models_checked=("model", "nunique"),
        latitude_counts=("latitude_count", lambda x: sorted(x.unique())),
        longitude_counts=("longitude_count", lambda x: sorted(x.unique())),
        time_steps=("time_steps", lambda x: sorted(x.unique())),
        calendars=("calendar", lambda x: sorted(x.dropna().unique())),
        units=("units", lambda x: sorted(x.dropna().unique()))
    )
    .reset_index()
)
print(metadata_summary.to_string(index=False))

#group models with identical spatial coordinates separately for each scenario and variable
spatial_groups = []
for scenario in scenarios:
    for variable in variables:
        grid_groups = {}
        for model in common_models:
            key = (model, scenario, variable)
            if key not in coordinate_data:
                continue
            lat, lon = coordinate_data[key]
            # Create a unique identifier for each grid
            grid_key = (
                lat.shape,
                lon.shape,
                lat.tobytes(),
                lon.tobytes()
            )
            if grid_key not in grid_groups:
                grid_groups[grid_key] = []
            grid_groups[grid_key].append(model)
        # Store each unique grid group
        for i, (grid, group_models) in enumerate(
            grid_groups.items(), 1
        ):
            spatial_groups.append({
                "scenario": scenario,
                "variable": variable,
                "grid_group": i,
                "number_of_models": len(group_models),
                "models": group_models
            })
spatial_groups_df = pd.DataFrame(spatial_groups)
print(spatial_groups_df.to_string(index=False))

#summarise spatial compatibility
spatial_summary = (
    spatial_groups_df
    .groupby(["scenario", "variable"])
    .agg(
        unique_grids=("grid_group", "nunique"),
        models_checked=("number_of_models", "sum")
    )
    .reset_index()
)
print(spatial_summary.to_string(index=False))


  scenario variable  models_checked latitude_counts longitude_counts time_steps                                         calendars        units
historical     hurs              27           [600]           [1440] [360, 365] [360_day, 365_day, proleptic_gregorian, standard]          [%]
historical       pr              27           [600]           [1440] [360, 365] [360_day, 365_day, proleptic_gregorian, standard] [kg m-2 s-1]
historical   tasmax              27           [600]           [1440] [360, 365] [360_day, 365_day, proleptic_gregorian, standard]          [K]
historical   tasmin              27           [600]           [1440] [360, 365] [360_day, 365_day, proleptic_gregorian, standard]          [K]
    ssp126     hurs              27           [600]           [1440] [360, 365] [360_day, 365_day, proleptic_gregorian, standard]          [%]
    ssp126       pr              27           [600]           [1440] [360, 365] [360_day, 365_day, proleptic_gregorian, standard] [kg m-2 s-1]